# CNN-LSTM baseline -- train + evaluate (Colab)

Trains the CNN-LSTM baseline (`ml_analytics/models/cnn_lstm.py`) on the frozen
HoneySynth dataset and saves predictions in the exact format
`ml_analytics/mt3_pipeline/compare.py` expects, so it can be compared head-to-head
against MT3.

**Self-contained**: clones the repo, unzips `honeysynth_final.zip`, trains, evaluates,
zips the results for download. Does **not** touch `ml_analytics/models/mt3.py`,
`ml_analytics/mt3_pipeline/`, `honeypot_dataset/`, or any dataset notebook (01-05, 03b).

**Note:** `ERRORS.md` is a local, per-machine file (see `TEAMMATES.md`) and was not
present in the checkout this notebook was built from, so its "top two entries" could
not be read ahead of time here -- if you have a copy, skim it before the first epoch.

**Headline metric:** macro-F1 on `X_test_real`, averaged over the **21 classes present**
in that split (not all 45 -- see the evaluation cell for why).

**Reference numbers** (same data/splits/scaler/class-weighting):

| model | params | val macro-F1 | test_real macro-F1 (headline) | test_synth macro-F1 |
|---|---|---|---|---|
| linear probe | 5,805 | 0.9376 | 0.7681 (floor -- below this = a bug, not a result) | 0.9373 |
| MT3 (d=256, 4 layers) | 3,759,510 | 0.9599 | 0.8276 | 0.9589 |


## 1. Environment

In [1]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU (T4), then re-run.")


torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM (GB): 15.6


In [2]:
!pip -q install -U scikit-learn joblib >/dev/null 2>&1
print("scikit-learn / joblib ready")


scikit-learn / joblib ready


## 2. Clone the repo (branch: `feature/Anushka_baseline-LSTM_CNN`)

That branch already has `ml_analytics/models/cnn_lstm.py` committed, so it's imported
directly rather than redefined in this notebook.

In [3]:
from pathlib import Path

REPO_URL = "https://github.com/MKD2004/adaptive-honeypot-ml-CAPSTONE.git"
BRANCH = "feature/Anushka_baseline-LSTM_CNN"
REPO_DIR = "adaptive-honeypot-ml-CAPSTONE"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path(REPO_DIR).exists():
        !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

REPO_ROOT = Path.cwd()
print("repo root:", REPO_ROOT)
!git branch --show-current
!git log --oneline -1


Cloning into 'adaptive-honeypot-ml-CAPSTONE'...
remote: Enumerating objects: 444, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 444 (delta 4), reused 13 (delta 4), pack-reused 427 (from 1)
Receiving objects: 100% (444/444), 1.16 MiB | 1.92 MiB/s, done.
Resolving deltas: 100% (189/189), done.
/content/adaptive-honeypot-ml-CAPSTONE
From https://github.com/MKD2004/adaptive-honeypot-ml-CAPSTONE
 * branch            feature/Anushka_baseline-LSTM_CNN -> FETCH_HEAD
Branch 'feature/Anushka_baseline-LSTM_CNN' set up to track remote branch 'feature/Anushka_baseline-LSTM_CNN' from 'origin'.
Switched to a new branch 'feature/Anushka_baseline-LSTM_CNN'
From https://github.com/MKD2004/adaptive-honeypot-ml-CAPSTONE
 * branch            feature/Anushka_baseline-LSTM_CNN -> FETCH_HEAD
Already up to date.
repo root: /content/adaptive-honeypot-ml-CAPSTONE
feature/Anushka_baseline-LSTM_CNN
acad939 (HEAD -> feature/Anushka_baseline-LSTM_C

## 3. Get the dataset

Looks for `honeysynth_final.zip` in a few likely places; if it's not found and this is
Colab, mounts Drive, then falls back to a manual upload prompt.

In [4]:
DATA_ZIP_CANDIDATES = [
    Path("/content/honeysynth_final.zip"),
    REPO_ROOT / "honeysynth_final.zip",
    Path("/content/drive/MyDrive/capstone/honeysynth_final.zip"),
]

zip_path = next((p for p in DATA_ZIP_CANDIDATES if p.exists()), None)

if zip_path is None and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        zip_path = next((p for p in DATA_ZIP_CANDIDATES if p.exists()), None)
    except Exception as exc:
        print("Drive mount skipped/failed:", exc)

if zip_path is None and IN_COLAB:
    print("honeysynth_final.zip not found automatically -- upload it now:")
    from google.colab import files
    uploaded = files.upload()
    zip_path = Path(next(iter(uploaded)))

if zip_path is None:
    raise FileNotFoundError(
        "honeysynth_final.zip not found. Place it at one of:\n  "
        + "\n  ".join(str(p) for p in DATA_ZIP_CANDIDATES)
        + "\nor upload it via the Colab file browser."
    )

print("using dataset zip:", zip_path, f"({zip_path.stat().st_size / 1e6:.0f} MB)")


Drive mount skipped/failed: Error: credential propagation was unsuccessful
honeysynth_final.zip not found automatically -- upload it now:


Saving honeysynth_final.zip to honeysynth_final.zip
using dataset zip: honeysynth_final.zip (196 MB)


In [5]:
import zipfile

DATA_DIR = REPO_ROOT / "honeypot_dataset" / "data" / "final"
DATA_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    assert not any("/" in n or n.startswith("..") for n in names), f"unexpected path in zip: {names}"
    z.extractall(DATA_DIR)

print(f"extracted {len(names)} files to {DATA_DIR}")
for p in sorted(DATA_DIR.iterdir()):
    print(f"  {p.stat().st_size:>14,}  {p.name}")


extracted 11 files to /content/adaptive-honeypot-ml-CAPSTONE/honeypot_dataset/data/final
      30,720,128  X_test_real.npy
      30,720,128  X_test_synth.npy
     387,072,128  X_train.npy
      43,008,128  X_val.npy
           1,030  dataset_card.json
           3,655  feature_scaler.pkl
           3,012  quality_report.json
         480,128  y_test_real.npy
         480,128  y_test_synth.npy
       6,048,128  y_train.npy
         672,128  y_val.npy


## 4. Load + sanity-check the frozen splits

Two silent-corruption traps from the prompt, checked explicitly below:
1. The arrays are **already scaled** -- `feature_scaler.pkl` is loaded for provenance
   only and is never applied to these arrays.
2. `feature_scaler.pkl` is a **joblib** dump, not a plain pickle.

In [6]:
import numpy as np
import joblib

def load(name):
    return np.load(DATA_DIR / f"{name}.npy")

X_train, y_train = load("X_train"), load("y_train")
X_val,   y_val   = load("X_val"),   load("y_val")
X_test_real,  y_test_real  = load("X_test_real"),  load("y_test_real")
X_test_synth, y_test_synth = load("X_test_synth"), load("y_test_synth")

scaler = joblib.load(DATA_DIR / "feature_scaler.pkl")  # provenance only -- do NOT call .transform() below
print("scaler:", type(scaler).__name__, "| n_features_in_:", scaler.n_features_in_,
      "| n_samples_seen_:", scaler.n_samples_seen_)

col_mean_abs = float(np.abs(X_train.mean(0)).mean())
nan_count = int(np.isnan(X_train).sum())

print()
print(f"X_train.shape          = {X_train.shape}   (expect (756000, 128))")
print(f"classes in y_train     = {len(np.unique(y_train))}   (expect 45)")
print(f"classes in y_test_real = {len(np.unique(y_test_real))}   (expect 21)")
print(f"mean(|col means|)      = {col_mean_abs:.4f}   (expect ~0.002 -- ~4.8 means double-scaled, RESTART)")
print(f"NaN count in X_train    = {nan_count}   (expect 0)")

assert X_train.shape == (756000, 128), "wrong file / partial unzip"
assert len(np.unique(y_train)) == 45, "wrong file -- did you load X_real.npy (22 cls) instead?"
assert len(np.unique(y_test_real)) == 21, "wrong test split loaded"
assert col_mean_abs < 0.5, "data looks unscaled, or already re-scaled -- do NOT call scaler.transform() on these arrays"
assert nan_count == 0, "corrupt transfer -- re-unzip honeysynth_final.zip"
print("\nAll sanity checks passed.")


scaler: StandardScaler | n_features_in_: 128 | n_samples_seen_: 756000.0

X_train.shape          = (756000, 128)   (expect (756000, 128))
classes in y_train     = 45   (expect 45)
classes in y_test_real = 21   (expect 21)
mean(|col means|)      = 0.0000   (expect ~0.002 -- ~4.8 means double-scaled, RESTART)
NaN count in X_train    = 0   (expect 0)

All sanity checks passed.


## 5. Model -- import the baseline (no DistilBERT transformer is run)

In [7]:
import sys
sys.path.insert(0, str(REPO_ROOT))

from ml_analytics.models.cnn_lstm import (
    CNNLSTM, FEATURE_GROUPS, DEAD_GROUPS, N_CLASSES, class_weights_from_labels,
)

print("architecture (from ml_analytics/models/cnn_lstm.py, mirrors ml_analytics/README.md):")
for name, g in FEATURE_GROUPS.items():
    dead = "  (constant-zero in this dataset)" if name in DEAD_GROUPS else ""
    print(f"  {g['arch']:<10} {name:<15} cols [{g['start']}:{g['end']}]{dead}")

_probe = CNNLSTM()
print(f"\nparameter count: {_probe.count_parameters():,}")
del _probe


architecture (from ml_analytics/models/cnn_lstm.py, mirrors ml_analytics/README.md):
  LSTM       A_temporal      cols [0:24]
  CNN        B_network       cols [24:52]
  CNN        C_payload       cols [52:76]
  DistilBERT D_semantic      cols [76:106]
  CNN+LSTM   E_threat_intel  cols [106:120]  (constant-zero in this dataset)
  CNN        F_tls_host      cols [120:128]  (constant-zero in this dataset)

parameter count: 189,581


## 6. Kill-chain phase mapping (local copy, mirrors `schema.py` / `mt3.py`)

`CNNLSTM` has no auxiliary phase head (MT3 does). `phase_pred` below is therefore
**derived** from the predicted micro-state via this lookup table, not from a separate
prediction -- that's the honest choice for a model that doesn't have a phase head, and
`compare.py`'s own phase macro-F1 is computed the same way (from `y_true`/`y_pred`),
so this doesn't disadvantage the comparison.

In [8]:
IDX_TO_PHASE = (
    [0] * 6   # 0-5   Reconnaissance
    + [1] * 6  # 6-11  Initial Access
    + [2] * 6  # 12-17 Execution
    + [3] * 5  # 18-22 Discovery
    + [4] * 4  # 23-26 Privilege Escalation
    + [5] * 5  # 27-31 Persistence
    + [6] * 5  # 32-36 Defense Evasion
    + [7] * 3  # 37-39 Lateral Movement
    + [8] * 5  # 40-44 Exfiltration
)
assert len(IDX_TO_PHASE) == N_CLASSES

def phase_of(y):
    lut = np.asarray(IDX_TO_PHASE, dtype=np.int64)
    return lut[y]


## 7. Train

Preloads the full train/val tensors onto the GPU once (they're ~387 MB / ~43 MB --
comfortably fits) and indexes batches directly, rather than using a `DataLoader`, so
the training loop isn't CPU-batching-bound. Class-weighted focal loss, gradient
clipping, `ReduceLROnPlateau` on val macro-F1, early stopping, checkpoint the best
val macro-F1 (not accuracy -- accuracy looks good even for a bad model here).

In [9]:
from sklearn.metrics import f1_score, accuracy_score
import time

EPOCHS = 100
BATCH_SIZE = 1024
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
PATIENCE = 15  # early-stop on val macro-F1 -- epoch 40/40 in the last run ended without
                # triggering the old PATIENCE=8, i.e. val_f1 was still climbing when the
                # run was cut off by EPOCHS, not by a real plateau. Widened both.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on:", device)

ARTIFACT_DIR = REPO_ROOT / "ml_analytics" / "artifacts" / "cnn_lstm"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

Xtr = torch.from_numpy(X_train).float().to(device)
ytr = torch.from_numpy(y_train).long().to(device)
Xva = torch.from_numpy(X_val).float().to(device)
yva = torch.from_numpy(y_val).long().to(device)
print(f"Xtr: {tuple(Xtr.shape)} on {Xtr.device}, {Xtr.element_size() * Xtr.nelement() / 1e6:.0f} MB")

class_weights = class_weights_from_labels(y_train, scheme="inverse_sqrt").to(device)
model = CNNLSTM(loss_type="focal", class_weights=class_weights, dropout=0.3).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)

# step-0 sanity check: an untrained 45-way classifier should score ~ln(45)
model.eval()
with torch.no_grad():
    _, l0 = model(Xtr[:512], ytr[:512])
print(f"loss at step 0: {l0.item():.4f}  (expect approx ln(45) = {np.log(45):.4f})")

n = Xtr.shape[0]
best_f1, best_state, bad_epochs = -1.0, None, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(n, device=device)
    tr_loss_sum, tr_correct = 0.0, 0
    t0 = time.time()
    for i in range(0, n, BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        xb, yb = Xtr[idx], ytr[idx]
        opt.zero_grad()
        logits, loss = model(xb, yb)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss at epoch {epoch}, batch {i} -- lower LR or check grad clip")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        tr_loss_sum += loss.item() * len(idx)
        tr_correct += (logits.argmax(1) == yb).sum().item()
    tr_loss, tr_acc = tr_loss_sum / n, tr_correct / n

    model.eval()
    with torch.no_grad():
        val_logits, _ = model(Xva)
        val_pred = val_logits.argmax(1).cpu().numpy()
    val_f1 = f1_score(y_val, val_pred, labels=np.unique(y_val), average="macro", zero_division=0)
    val_acc = accuracy_score(y_val, val_pred)
    sched.step(val_f1)

    flags = []
    if tr_acc - val_acc > 0.15:
        flags.append("growing train/val accuracy gap -- possible overfitting")
    if val_f1 < 0.05 and val_acc > 0.2:
        flags.append("macro-F1 near zero while accuracy is high -- class weighting may not be applied")

    dt = time.time() - t0
    print(f"epoch {epoch:3d}/{EPOCHS}  loss {tr_loss:.4f}  train_acc {tr_acc:.4f}  "
          f"val_f1 {val_f1:.4f}  val_acc {val_acc:.4f}  {dt:.1f}s"
          + (f"  (best, {bad_epochs}->0)" if val_f1 > best_f1 else f"  (no improvement, {bad_epochs + 1}/{PATIENCE})"))
    for f in flags:
        print(f"  [warn] {f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
        torch.save(best_state, ARTIFACT_DIR / "checkpoint_best.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stop at epoch {epoch} (no val macro-F1 improvement for {PATIENCE} epochs)")
            break
else:
    print(f"\nreached EPOCHS={EPOCHS} without early stopping -- val_f1 may still be improving; "
          f"consider raising EPOCHS again if the last few epochs were still climbing.")

print(f"\nbest val macro-F1: {best_f1:.4f}")
model.load_state_dict(best_state)


training on: cuda
Xtr: (756000, 128) on cuda:0, 387 MB
loss at step 0: 3.6626  (expect approx ln(45) = 3.8067)
epoch   1/100  loss 0.3121  train_acc 0.9316  val_f1 0.9443  val_acc 0.9692  7.0s  (best, 0->0)
epoch   2/100  loss 0.1138  train_acc 0.9661  val_f1 0.9509  val_acc 0.9731  6.9s  (best, 0->0)
epoch   3/100  loss 0.0977  train_acc 0.9688  val_f1 0.9522  val_acc 0.9738  6.4s  (best, 0->0)
epoch   4/100  loss 0.0908  train_acc 0.9697  val_f1 0.9534  val_acc 0.9744  6.8s  (best, 0->0)
epoch   5/100  loss 0.0867  train_acc 0.9704  val_f1 0.9515  val_acc 0.9735  6.7s  (no improvement, 1/15)
epoch   6/100  loss 0.0832  train_acc 0.9712  val_f1 0.9540  val_acc 0.9750  6.5s  (best, 1->0)
epoch   7/100  loss 0.0812  train_acc 0.9712  val_f1 0.9539  val_acc 0.9751  7.0s  (no improvement, 1/15)
epoch   8/100  loss 0.0791  train_acc 0.9717  val_f1 0.9552  val_acc 0.9755  6.3s  (best, 1->0)
epoch   9/100  loss 0.0769  train_acc 0.9722  val_f1 0.9564  val_acc 0.9763  7.0s  (best, 0->0)
epoch

<All keys matched successfully>

## 8. Evaluate + save predictions

Saves `preds_test_real.npz` / `preds_test_synth.npz` in the exact format
`compare.py` expects. **Headline macro-F1 is averaged over the classes present in
`y_true`** (21 for `test_real`) -- averaging over all 45 would score the 24 absent
classes as 0 by construction, which is not a performance number (the prompt calls
this out explicitly: that mistake lands you at ~0.386).

In [10]:
def evaluate_split(X, y, split_name):
    model.eval()
    Xd = torch.from_numpy(X).float().to(device)
    preds = []
    with torch.no_grad():
        for i in range(0, len(Xd), 4096):
            logits, _ = model(Xd[i:i + 4096])
            preds.append(logits.argmax(1).cpu().numpy())
    y_pred = np.concatenate(preds).astype(np.int64)
    y_true = np.asarray(y).astype(np.int64)  # on-disk order, never shuffled

    present = np.unique(y_true)
    macro_f1_present = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    macro_f1_all45 = f1_score(y_true, y_pred, labels=np.arange(N_CLASSES), average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    phase_pred = phase_of(y_pred)  # derived -- no separate phase head on this model
    phase_true = phase_of(y_true)
    phase_f1 = f1_score(phase_true, phase_pred, labels=np.unique(phase_true), average="macro", zero_division=0)

    print(f"\n{split_name}: n={len(y_true)}  classes_present={len(present)}/{N_CLASSES}")
    print(f"  accuracy                    = {acc:.4f}")
    print(f"  weighted-F1                 = {weighted_f1:.4f}")
    print(f"  macro-F1 ({len(present)} present classes) = {macro_f1_present:.4f}   <-- headline")
    print(f"  macro-F1 (all 45)           = {macro_f1_all45:.4f}   (NOT the headline)")
    print(f"  phase macro-F1 (9)          = {phase_f1:.4f}")

    out_path = ARTIFACT_DIR / f"preds_{split_name}.npz"
    np.savez_compressed(
        out_path,
        y_true=y_true, y_pred=y_pred, phase_pred=phase_pred,
        model=np.array("cnn_lstm"), split=np.array(split_name),
    )
    print(f"  saved -> {out_path}")
    return {"macro_f1_present": macro_f1_present, "macro_f1_all45": macro_f1_all45,
            "accuracy": acc, "weighted_f1": weighted_f1, "phase_macro_f1": phase_f1}

res_real = evaluate_split(X_test_real, y_test_real, "test_real")
res_synth = evaluate_split(X_test_synth, y_test_synth, "test_synth")

print("\n" + "=" * 64)
print("REFERENCE CHECK (same data/splits/scaler/class-weighting):")
print("  linear probe (5,805 params)      test_real macro-F1 = 0.7681  (floor)")
print("  MT3 (3,759,510 params)           test_real macro-F1 = 0.8276")
print(f"  THIS MODEL ({model.count_parameters():,} params)     test_real macro-F1 = {res_real['macro_f1_present']:.4f}")
if res_real["macro_f1_present"] < 0.7681:
    print("  [WARNING] below the linear-probe floor -- this points to a training-setup bug, not a modeling result.")
else:
    print("  Above the linear-probe floor.")



test_real: n=60000  classes_present=21/45
  accuracy                    = 0.9956
  weighted-F1                 = 0.9959
  macro-F1 (21 present classes) = 0.8093   <-- headline
  macro-F1 (all 45)           = 0.3777   (NOT the headline)
  phase macro-F1 (9)          = 0.8372
  saved -> /content/adaptive-honeypot-ml-CAPSTONE/ml_analytics/artifacts/cnn_lstm/preds_test_real.npz

test_synth: n=60000  classes_present=45/45
  accuracy                    = 0.9740
  weighted-F1                 = 0.9742
  macro-F1 (45 present classes) = 0.9610   <-- headline
  macro-F1 (all 45)           = 0.9610   (NOT the headline)
  phase macro-F1 (9)          = 0.9696
  saved -> /content/adaptive-honeypot-ml-CAPSTONE/ml_analytics/artifacts/cnn_lstm/preds_test_synth.npz

REFERENCE CHECK (same data/splits/scaler/class-weighting):
  linear probe (5,805 params)      test_real macro-F1 = 0.7681  (floor)
  MT3 (3,759,510 params)           test_real macro-F1 = 0.8276
  THIS MODEL (189,581 params)     test_real mac

## 9. Package results for transfer

`ml_analytics/artifacts/` is git-ignored, so these files never land in git history.
Zips the artifact dir and, in Colab, triggers a download.

In [11]:
import shutil

zip_out = shutil.make_archive(str(REPO_ROOT / "cnn_lstm_artifacts"), "zip", root_dir=ARTIFACT_DIR)
print("artifacts zipped:", zip_out)

if IN_COLAB:
    from google.colab import files
    files.download(zip_out)
else:
    print("Not running in Colab -- artifacts are at:", ARTIFACT_DIR)


artifacts zipped: /content/adaptive-honeypot-ml-CAPSTONE/cnn_lstm_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Next step -- compare against MT3

Unzip `cnn_lstm_artifacts.zip` into `ml_analytics/artifacts/cnn_lstm/` on whichever
machine holds MT3's predictions (`ml_analytics/artifacts/mt3_full_d256/`), then:

```bash
python -m ml_analytics.mt3_pipeline.compare \
    --mt3-dir      ml_analytics/artifacts/mt3_full_d256 \
    --baseline-dir ml_analytics/artifacts/cnn_lstm
```

It verifies both models were scored on identical rows (aborts if not), prints the
head-to-head table with per-class F1 deltas, and runs McNemar's test for statistical
significance.

**Not done by this notebook:** `ml_analytics/models/model_trainer.py` is still empty
on `main` -- this training loop lives only here. If you want it promoted into a
reusable trainer module, that's a separate step.